# Train/Test Split for the Repair Experiments

To compare the repair strategies (R0–R4) fairly, we split the queries into a **train** set (for developing and tuning strategies) and a **test** set (held out for the final comparison). Tuning on the whole dataset would inflate the final numbers, so the test set stays untouched until the end.

All cells run offline (no API calls, no cost) and produce real output, so the split is fully reproducible.

**Properties of the split**

- Deterministic (fixed seed) — everyone gets the same train/test sets.
- No leakage — a query is never in both sides.
- Wired into the runner — pass the split name via `--splits`.

**Two splits are available:**

1. **Stratified combined split** (used for the experiments) — a 70/30 split taken within each of `easy`, `medium`, `human`, so both sides share the same difficulty mix (~50% easy, 25% medium, 25% human). Use `--splits combined_train` / `--splits combined_test`. `preference` queries are excluded (they're scored on soft goals, not pass/fail).
2. **`medium`-only split** (`--splits subset_train` / `medium_test`) — the simplest worked example of the split logic.

Steps 1–5 use the `medium` example; Step 5b is the stratified split the experiments actually use.

## Step 1: What we're splitting

Queries come from the bilingual ChinaTravel dataset on Hugging Face. Runs are in English, so each query's English text (`nature_language_en`) is used and city names are translated to English (handled in `chinatravel/data/load_datasets.py`). The split logic is in `chinatravel/evaluation/train_test_split.py` (`build_split`).

Load the full `medium` split to see how many queries there are.

In [1]:
import os, io, contextlib
from types import SimpleNamespace

# Run from the repo root so the chinatravel package imports resolve.
if os.path.basename(os.getcwd()) != 'Agentic-Travel-Planner-Ablation-of-Verifications-' \
        and os.path.isdir('Agentic-Travel-Planner-Ablation-of-Verifications-'):
    os.chdir('Agentic-Travel-Planner-Ablation-of-Verifications-')

from chinatravel.data.load_datasets import load_query

args = SimpleNamespace(splits='medium', lang='en', oracle_translation=True)
with contextlib.redirect_stdout(io.StringIO()):  # hide the loader's own logging
    ids, data = load_query(args)
records = [data[u] for u in ids]

print('Total queries in the medium split:', len(records))
print('Example query:', records[0]['start_city'], '->', records[0]['target_city'])
print('  ', records[0]['nature_language'][:120])

Total queries in the medium split: 150
Example query: Shanghai -> Beijing
   Current location: Shanghai. My friend and I are planning to visit Beijing for three days with a budget of 6000 RMB. We w


## Step 2: Make the split

A 70/30 split with a fixed seed (42). The seed makes it deterministic, so rebuilding gives the same train/test sets. `build_split` groups related queries, checks for leakage across the two sides, and validates the manifest. The output below confirms the sizes, the fraction, and that there's no overlap.

In [2]:
from chinatravel.evaluation.train_test_split import build_split, validate_manifest

manifest = build_split(records, train_fraction=0.7, seed=42)
validate_manifest(manifest)  # raises if anything is inconsistent

print('train queries :', len(manifest['train']))
print('test queries  :', len(manifest['test']))
print('train fraction : {:.2f}'.format(len(manifest['train']) / len(records)))
print('overlap        :', set(manifest['train']) & set(manifest['test']) or 'none')
print('seed / algorithm:', manifest['seed'], '/', manifest['algorithm'])

train queries : 105
test queries  : 45
train fraction : 0.70
overlap        : none
seed / algorithm: 42 / sha256-seed-group-ranking-v1


## Step 3: Check it's reproducible

Rebuilding with the same seed gives an identical split; a different seed gives a different one. This confirms the split is stable across machines, not random per run.

In [3]:
same = build_split(records, train_fraction=0.7, seed=42)
diff = build_split(records, train_fraction=0.7, seed=7)

print('same seed -> identical split :',
      same['train'] == manifest['train'] and same['test'] == manifest['test'])
print('different seed -> different split:', diff['train'] != manifest['train'])

same seed -> identical split : True
different seed -> different split: True


## Step 4: Save the split and wire it into the runner

A "split" in ChinaTravel is just a text file of query UIDs under `chinatravel/evaluation/default_splits/`. The train/test UID lists are saved as `medium_train.txt` / `medium_test.txt`, and the full manifest (with audit info) as JSON under `chinatravel/evaluation/splits/`.

Since the query text lives on Hugging Face (not local files), `load_query` in `load_datasets.py` was extended so a name like `medium_train` loads the base `medium` queries from Hugging Face and keeps only the UIDs in `medium_train.txt` — so `--splits subset_train` works with no local query files.

The cell below (re)writes the split files so the notebook is self-contained.

In [4]:
import json

# 1) Manifest JSON with the full audit trail.
os.makedirs('chinatravel/evaluation/splits', exist_ok=True)
mpath = 'chinatravel/evaluation/splits/medium_train_test_seed42.json'
with open(mpath, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, sort_keys=True)
print('wrote', mpath)

# 2) The two UID lists the runner consumes via --splits.
for name, key in [('medium_train', 'train'), ('medium_test', 'test')]:
    p = f'chinatravel/evaluation/default_splits/{name}.txt'
    with open(p, 'w', encoding='utf-8') as f:
        f.write('\n'.join(manifest[key]) + '\n')
    print(f'wrote {p} ({len(manifest[key])} uids)')

wrote chinatravel/evaluation/splits/medium_train_test_seed42.json
wrote chinatravel/evaluation/default_splits/medium_train.txt (105 uids)
wrote chinatravel/evaluation/default_splits/medium_test.txt (45 uids)


## Step 5: Check the subsets load through the runner's loader

Load `medium_train` and `medium_test` through the same `load_query` the runner uses, and confirm they're the right sizes, disjoint, and together reconstruct the full `medium` split. One train query is printed to show it loads in English.

In [5]:
def load(split):
    a = SimpleNamespace(splits=split, lang='en', oracle_translation=True)
    with contextlib.redirect_stdout(io.StringIO()):
        return load_query(a)

tr_ids, tr = load('medium_train')
te_ids, te = load('medium_test')
full_ids, _ = load('medium')

print('medium_train:', len(tr_ids))
print('medium_test :', len(te_ids))
print('medium (all):', len(full_ids))
print('train/test disjoint        :', set(tr_ids).isdisjoint(te_ids))
print('train + test == full medium:', set(tr_ids) | set(te_ids) == set(full_ids))
print()
q = tr[tr_ids[0]]
print('example train query (English):', q['start_city'], '->', q['target_city'])
print('  ', q['nature_language'][:140])

medium_train: 105
medium_test : 45
medium (all): 150
train/test disjoint        : True
train + test == full medium: True

example train query (English): Guangzhou -> Hangzhou
   Current location: Guangzhou. Four of us are planning to visit Hangzhou for two days, traveling by train, with a budget of 5600 RMB. We will 


## Step 5b: The stratified combined split (used for the experiments)

For the experiments we cover more than one difficulty level, so the combined split spans `easy`, `medium`, and `human`. The same 70/30 logic (seed 42) is applied **within each category** and then merged — this keeps train and test at the same difficulty mix. Splitting the pooled set blindly could push the harder `human` queries disproportionately onto one side and skew the comparison.

Since these come from different Hugging Face splits, a small map (`chinatravel/evaluation/splits/combined_origin.json`) records each query's source split, and the loader uses it so `--splits combined_train` / `--splits combined_test` fetch each query from the right place. `preference` queries are excluded (scored on soft goals, not pass/fail).

The cell below loads both sides and shows the sizes, that they're disjoint, and that both keep the same difficulty mix.

In [6]:
import json
from collections import Counter

origin = json.load(open('chinatravel/evaluation/splits/combined_origin.json'))['origin']

with contextlib.redirect_stdout(io.StringIO()):
    ctr_ids, _ = load_query(SimpleNamespace(splits='combined_train', lang='en', oracle_translation=True))
    cte_ids, _ = load_query(SimpleNamespace(splits='combined_test',  lang='en', oracle_translation=True))

def mix(ids):
    c = Counter(origin[u] for u in ids); t = len(ids)
    return {k: f'{100*v/t:.0f}%' for k, v in sorted(c.items())}

print('combined_train:', len(ctr_ids))
print('combined_test :', len(cte_ids))
print('disjoint (no leakage):', set(ctr_ids).isdisjoint(cte_ids))
print('train difficulty mix:', mix(ctr_ids))
print('test  difficulty mix:', mix(cte_ids))

combined_train: 422
combined_test : 182
disjoint (no leakage): True
train difficulty mix: {'easy': '50%', 'human': '25%', 'medium': '25%'}
test  difficulty mix: {'easy': '49%', 'human': '26%', 'medium': '25%'}


## Step 6: Running R0 on the split (the experiment)

R0 runs on each query, saving one result file per query. The cells below show how a single query is run and then loop over the sample, printing each query's outcome and difficulty.

**Reproducibility note.** R0 calls the LLM (DeepSeek V3), which costs money and is non-deterministic (two runs of the same query give different itineraries). So results are **run once and committed** to `experiments/r0_combined_sample/` (a tracked folder, unlike the git-ignored `results/`). Re-running the notebook **loads** the committed files instead of calling the API — no repeat cost, identical files for everyone.

Live calls require `RUN_LIVE = True` and an OpenRouter key in `.env`. It defaults to `False` so the notebook never spends credits by accident.

**Committed here:** a **5-query sample spanning the combined split** (2 easy, 1 medium, 2 human), so the sample reflects the difficulty mix the experiments use. Running the full split is the same procedure with `MAX_QUERIES = None`. An earlier medium-only sample is kept under `experiments/r0_medium_sample/` for reference.

> **Earlier bug (fixed):** the first attempt produced empty itineraries — a library mismatch (`openai` using `httpx2` while the usage-instrumentation transport used the old `httpx`) made every model call fail silently. Fixed in `chinatravel/agent/model_usage.py`; the results below are from the working version.

In [7]:
import os, io, json, contextlib
from types import SimpleNamespace

# Committed R0 results live in a tracked folder (unlike the git-ignored results/).
# The primary sample spans the combined split (easy/medium/human); the medium-only
# sample from an earlier run is kept separately for reference.
RESULTS_DIR = 'experiments/r0_combined_sample'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Safety switch: keep False so running the notebook never makes paid API calls.
# Set True (with an OpenRouter key in .env) to run live.
RUN_LIVE = False

# Cap on how many queries to run live, for a cheap sample. None = whole split.
# Loading already-committed results is never capped (reading files is free).
MAX_QUERIES = 5

# Which split to draw queries from when running live.
SPLIT = 'subset_train'

# Fixed experiment settings for the R0 baseline (same budget for every strategy).
REPAIR_STRATEGY = 'R0'
REFINE_STEPS = 5
LLM = 'deepseek/deepseek-chat'  # DeepSeek V3 via OpenRouter

# Difficulty of each query (easy/medium/human), for slicing results by difficulty.
ORIGIN = json.load(open('chinatravel/evaluation/splits/combined_origin.json'))['origin']

def result_path(uid):
    return os.path.join(RESULTS_DIR, f'{uid}.json')

print('results dir :', RESULTS_DIR)
print('RUN_LIVE    :', RUN_LIVE, '| MAX_QUERIES:', MAX_QUERIES, '| SPLIT:', SPLIT)
print('strategy    :', REPAIR_STRATEGY, '| refine_steps:', REFINE_STEPS, '| llm:', LLM)

results dir : experiments/r0_combined_sample
RUN_LIVE    : False | MAX_QUERIES: 5 | SPLIT: subset_train
strategy    : R0 | refine_steps: 5 | llm: deepseek/deepseek-chat


### 6.1 How one query is run

The recipe for a single query mirrors `run_exp.py`: build the agent once, call `agent.solve(query, ...)`, save the plan as `<uid>.json`. The equivalent command for the whole split:

```bash
python run_exp.py \
  --agent LLM-modulo \
  --repair_strategy R0 \
  --llm deepseek/deepseek-chat \
  --splits subset_train \
  --refine_steps 5 \
  --oracle_translation
```

The function below is the per-query version of that command.

In [8]:
def build_runtime():
    """Build the R0 agent runtime once (same as run_exp.py sets up)."""
    from chinatravel.agent.load_model import create_agent_runtime
    return create_agent_runtime(
        'LLM-modulo', LLM,
        project_root_path=os.getcwd(),
        lang='en', oracle_translation=True,
        refine_steps=REFINE_STEPS, repair_strategy=REPAIR_STRATEGY,
    )

def run_one_query(runtime, uid, query):
    """Run R0 on one query and save the plan. Returns the plan dict."""
    succ, plan = runtime.agent.solve(query, prob_idx=uid, oracle_verifier=True)
    with open(result_path(uid), 'w', encoding='utf-8') as f:
        json.dump(plan, f, ensure_ascii=False, indent=2)
    return plan

def summarise(uid, plan):
    """One-line outcome for a query's result."""
    errs = plan.get('error_info', [])
    pc = plan.get('personal_constraints', [])
    passed = (len(errs) == 0) and all(x is True for x in pc) if pc else (len(errs) == 0)
    return (f"{uid}: {'PASS' if passed else 'needs work'} | "
            f"reported_errors={len(errs)} | personal={pc}")

print('Helpers defined: build_runtime(), run_one_query(), summarise().')

Helpers defined: build_runtime(), run_one_query(), summarise().


### 6.2 Run (or load) each query

For each query the loop loads its committed result if present (no API call), or runs it live if `RUN_LIVE` is on. Each line shows the query's **difficulty** so the easy/medium/human spread is visible.

In [9]:
# Committed results drive the default (offline) view; the live path draws from SPLIT.
committed = sorted(f[:-5] for f in os.listdir(RESULTS_DIR) if f.endswith('.json'))

if RUN_LIVE:
    with contextlib.redirect_stdout(io.StringIO()):
        split_ids, split_data = load_query(
            SimpleNamespace(splits=SPLIT, lang='en', oracle_translation=True))
    # Run committed ones first (loaded), then fill up to MAX_QUERIES from the split.
    order = committed + [u for u in split_ids if u not in committed]
else:
    order, split_data = committed, {}

runtime = None
results = {}
n_loaded = n_ran = n_pending = 0

for i, uid in enumerate(order, start=1):
    diff = ORIGIN.get(uid, '?')
    if os.path.exists(result_path(uid)):
        with open(result_path(uid), encoding='utf-8') as f:
            plan = json.load(f)
        results[uid] = plan; n_loaded += 1
        print(f'[{i:>2}] loaded  [{diff:6}] {summarise(uid, plan)}')
    elif RUN_LIVE and n_ran < MAX_QUERIES:
        if runtime is None:
            with contextlib.redirect_stdout(io.StringIO()):
                runtime = build_runtime()
        with contextlib.redirect_stdout(io.StringIO()):
            plan = run_one_query(runtime, uid, split_data[uid])
        results[uid] = plan; n_ran += 1
        print(f'[{i:>2}] ran     [{diff:6}] {summarise(uid, plan)}')
    # (queries beyond the committed sample / cap are simply not shown)

print()
print(f'loaded from committed results : {n_loaded}')
print(f'ran live this session         : {n_ran}')

[ 1] loaded  [easy  ] e20241028160255878579: needs work | reported_errors=9 | personal=[True, True, True, True, True]
[ 2] loaded  [easy  ] e20241028160300570125: needs work | reported_errors=7 | personal=[True, True, True, True, True]
[ 3] loaded  [medium] e20241028160848776495: needs work | reported_errors=13 | personal=[True, True, False, True, True, False, True, True]
[ 4] loaded  [human ] h20241029143451793119: needs work | reported_errors=23 | personal=[False, True, True, True]
[ 5] loaded  [human ] h20241029143453343691: needs work | reported_errors=21 | personal=[True, True, True]

loaded from committed results : 5
ran live this session         : 0


### 6.3 Aggregate result, broken down by difficulty

A summary over the sample: overall pass rate and average errors, then the same split by difficulty. The difficulty breakdown is the interesting part — it shows whether R0 struggles more as queries get harder.

In [10]:
if results:
    def is_pass(plan):
        errs = plan.get('error_info', [])
        pc = plan.get('personal_constraints', [])
        return (len(errs) == 0) and (all(x is True for x in pc) if pc else True)

    total = len(results)
    n_pass = sum(1 for p in results.values() if is_pass(p))
    avg_err = sum(len(p.get('error_info', [])) for p in results.values()) / total
    print(f'queries with a result : {total}')
    print(f'passed                : {n_pass} ({100*n_pass/total:.0f}%)')
    print(f'avg reported errors   : {avg_err:.1f}')

    # Break the sample down by difficulty (easy/medium/human).
    print()
    print('by difficulty:')
    from collections import defaultdict
    buckets = defaultdict(list)
    for uid, plan in results.items():
        buckets[ORIGIN.get(uid, '?')].append(plan)
    for diff in ['easy', 'medium', 'human']:
        ps = buckets.get(diff, [])
        if not ps:
            continue
        passed = sum(1 for p in ps if is_pass(p))
        ae = sum(len(p.get('error_info', [])) for p in ps) / len(ps)
        print(f'  {diff:6}: {len(ps)} queries | passed {passed}/{len(ps)} | avg errors {ae:.1f}')
else:
    print('No committed results found. Set RUN_LIVE=True (with an API key) to produce them,')
    print('then commit experiments/r0_combined_sample/ so teammates get the same files.')

queries with a result : 5
passed                : 0 (0%)
avg reported errors   : 14.6

by difficulty:
  easy  : 2 queries | passed 0/2 | avg errors 8.0
  medium: 1 queries | passed 0/1 | avg errors 13.0
  human : 2 queries | passed 0/2 | avg errors 22.0


### 6.3b Reading the results (what "pass" means, and the difficulty trend)

**"Pass" is all-or-nothing.** A plan passes only if it satisfies *every* rule — the user's personal requests (budget, cuisine, room type…) **and** the commonsense rules (times line up, transport legs are real). One broken detail fails the whole plan.

**The difficulty trend is the key finding.** Looking at the breakdown above: the **easy** queries satisfy all their personal constraints, while **medium** and **human** queries break more and carry more reported errors. Error counts rise with complexity. This is direct evidence for the project's hypothesis — harder, more interdependent queries are where repair matters most, and where R1–R4 should show the biggest improvement over R0.

**The common failure is inner-city transport.** Most reported errors are `"Unavailable Inner-City Transport information"`. This isn't missing data: R0 lets the model write transport details between places, and the model guesses some wrong; when the verifier re-checks each leg against the real data, the mismatch is rejected. The cell below shows a concrete case — a correct metro route with a **made-up fare** (12 vs the real 4), enough to reject the leg.

**Why a low pass rate is expected.** R0 is the naive baseline — it regenerates whole plans without carefully grounding transport. Its struggles on harder queries are exactly the gap R1–R4 aim to close.

In [11]:
import io, contextlib, json, glob
from collections import Counter
from chinatravel.environment.world_env import WorldEnv

# (1) What kind of error dominates across the committed results?
kinds, total = Counter(), 0
for p in glob.glob('experiments/r0_combined_sample/*.json'):
    for e in json.load(open(p)).get('error_info', []):
        total += 1
        kinds['inner-city transport' if 'Inner-City Transport' in str(e) else 'other'] += 1
print('Error breakdown across the committed results:')
for k, v in kinds.most_common():
    print(f'  {k}: {v}  ({100*v/total:.0f}% of all {total} errors)')

# (2) One concrete example: a metro leg whose details DON'T match the real data.
# We scan committed plans for a metro leg where the fare the model wrote differs
# from the real sandbox fare for the same trip -- that mismatch is what the
# verifier rejects.
with contextlib.redirect_stdout(io.StringIO()):
    env = WorldEnv(lang='en')

def find_fare_mismatch():
    for p in sorted(glob.glob('experiments/r0_combined_sample/*.json')):
        d = json.load(open(p)); city = d.get('target_city')
        for day in d.get('itinerary', []):
            for act in day.get('activities', []):
                legs = act.get('transports')
                if not isinstance(legs, list) or not act.get('position'):
                    continue
                metro = [l for l in legs if l.get('mode') == 'metro']
                if not metro:
                    continue
                try:
                    with contextlib.redirect_stdout(io.StringIO()):
                        real = env.transportation.goto(city, legs[0]['start'], act['position'], legs[0]['start_time'], 'metro')
                    rm = [l for l in real if l.get('mode') == 'metro']
                except Exception:
                    continue
                if rm and rm[0].get('cost') != metro[0].get('cost'):
                    return city, legs[0]['start'], act['position'], metro[0]['cost'], rm[0]['cost']
    return None

ex = find_fare_mismatch()
if ex:
    city, origin, dest, llm_cost, real_cost = ex
    print(f'\nExample of a wrong detail ({city}): {origin} -> {dest}')
    print(f'  model wrote metro fare : {llm_cost}')
    print(f'  real sandbox fare      : {real_cost}')
    print(f'\n  -> Same route, but the model GUESSED the fare ({llm_cost}) while the real')
    print(f'     fare is {real_cost}. The verifier compares the plan to the real data, sees')
    print("     the mismatch, and rejects the leg. Enough rejected legs -> 'not passed'.")
else:
    print('\n(No fare mismatch found in this small sample; the transport errors here')
    print(' come from other guessed details such as timing or mode.)')


Error breakdown across the committed results:
  inner-city transport: 62  (85% of all 73 errors)
  other: 11  (15% of all 73 errors)



Example of a wrong detail (Hangzhou): Hangzhou East Railway Station -> Hangzhou West Lake Zhongwei Xiangyi Hotel
  model wrote metro fare : 48
  real sandbox fare      : 3

  -> Same route, but the model GUESSED the fare (48) while the real
     fare is 3. The verifier compares the plan to the real data, sees
     the mismatch, and rejects the leg. Enough rejected legs -> 'not passed'.


### 6.4 How to continue from here

- **Reproduce these results:** pull the repo (the committed results in `experiments/r0_combined_sample/` come with it) and run this notebook top to bottom — Step 6.2 loads them and Step 6.3 shows the same numbers, no API calls.
- **Extend the R0 run:** set `RUN_LIVE = True`, add an OpenRouter key to `.env`, and raise `MAX_QUERIES` (or set `None` for the whole split). Commit the new result files so the team gets them.
- **Run your own strategy (R1–R4):** the shared pipeline (`run_repair_comparison.py`) is the proper path for the full comparison — it runs every strategy through the same initial plan and logs standardized rows to `attempts.jsonl`. Keep `--refine_steps 5` and the same split across strategies. Each logged attempt's difficulty is recoverable from `combined_origin.json`, so the interaction matrix can be sliced by difficulty.

The committed `experiments/r0_combined_sample/*.json` files are the shared source of truth for this R0 sample; LLM output is non-deterministic, so fresh runs won't match them exactly.